# 9. Is the moving-object error inside the objects, or along their outlines? (CPU, high RAM)

Moving objects score worse than parked ones. That can be the model, or it can be the ground truth: LiDAR points of
an object that crosses the view land a few pixels beside it in the image, and that error sits on the object's
OUTLINE. This notebook splits every object's LiDAR pixels into boundary and interior and compares moving with
parked objects in the interior only, where misregistration cannot reach (see `src/vggt_aura/edges.py`). Parked
objects are the control for what soft depth edges alone cost.

Reads saved predictions and saved ground truth. Downloads only the small base layer of each block (boxes and
poses). Run on a **CPU high-RAM** server (8 cores): about 100 s of work per scene, 4 to 6 minutes per block, about
25 minutes for the six test blocks. Finished blocks are skipped.

In [1]:
# --- 1. Configuration ---
PERSIST_MODE = "drive"
DRIVE_ROOT = "/content/drive/MyDrive/vggt-omega-aura-benchmark"   # where predictions, ground truth and results live.
# Work already saved there is skipped. To run EVERYTHING again from the images up, name an empty folder here,
# the same one in every notebook of the run. The Hugging Face token is still found in the usual folder's .env.
RUN_TAG = "phase7_front_medium"       # the folder of this run in persistent storage. A name from the time of the work:
                                      # every notebook of the run must use the same one, the results live under it
CAMERA = "front_medium"
MODELS = ["vggt_omega_512", "vggt_1b"]
BLOCKS = [("test", 0), ("test", 1), ("test", 2), ("test", 3), ("test", 11), ("test", 12)]
# The test split: all daytime, so the ground truth is not disturbed by rain spray. Other blocks can be added.
RADII = [6, 12]                       # width of the boundary rim in pixels; two values show how much the answer depends on it
WORKERS = None                        # None = one per CPU core (max 8)

In [ ]:
# === CODE SYNC (auto-generated by `python -m vggt_aura.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m vggt_aura.sync   and reopen this notebook.")

In [3]:
# --- 3. Session ---
from vggt_aura.session import start_session

session = start_session(persist_mode=PERSIST_MODE, drive_root=DRIVE_ROOT, require_gpu=False)

GPU: none (fine for download and inspection)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
persist root: /content/drive/MyDrive/vggt-omega-aura-benchmark
data root   : /content/data/fzi-aura (runtime disk, wiped at session end)


In [4]:
# --- 4. Per block: fetch the base layer, split every scene's object pixels, save ---
import shutil, time
import pandas as pd
from vggt_aura import aura_data as ad, edges, pipeline as pl

pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 90)
chunks, scene_blocks, hub_files = ad.fetch_release_tables(session.data_root / "_release_tables")
EXCLUDED = ad.fetch_excluded_scene_ids(session.data_root / "_release_tables")
out_dir = session.persist_root / "metrics" / RUN_TAG / "edges"
out_dir.mkdir(parents=True, exist_ok=True)
tag = "r" + "-".join(str(r) for r in RADII)

for split, block in BLOCKS:
    target = out_dir / f"{pl.block_tag(split, block)}_{tag}.csv"
    if target.is_file():
        print(f"=== {pl.block_tag(split, block)}: already done ===")
        continue
    started = time.time()
    print(f"=== {pl.block_tag(split, block)} ===")
    data_root = session.data_root / f"edges_{pl.block_tag(split, block)}"
    scene_ids = ad.block_scene_ids(scene_blocks, split, block, EXCLUDED)
    if not ad.block_on_disk(data_root, scene_ids, []):
        ad.download_block(data_root, split, block, scene_ids, [])            # [] = the base layer only
    jobs = [{"scene_id": scene_id, "camera_id": CAMERA, "models": list(MODELS), "split": split, "block": block,
             "data_root": str(data_root), "persist_root": str(session.persist_root), "radii": list(RADII)} for scene_id in scene_ids]
    results = pl.run_jobs(jobs, WORKERS, function=pl.edge_scene,
                          on_result=lambda r: print(f"  {r['scene_id']:26} {r['seconds']:5.1f} s | " + " | ".join(r["notes"])))
    tables = [r["rows"] for r in results if len(r["rows"])]
    if tables:
        pd.concat(tables, ignore_index=True).to_csv(target.with_suffix(".tmp"), index=False)
        target.with_suffix(".tmp").replace(target)
    shutil.rmtree(data_root, ignore_errors=True)
    print(f"  {len(tables)} scenes with objects, {time.time() - started:.0f} s")

=== test_block000000 ===
  downloading with the toolkit, decompressing with xz on all 8 cores


  fast unpack: {'archives': 1, 'xz_decompressed_on_all_cores': 0, 'download_s': 8.3, 'verify_and_decompress_s': 9.7, 'extract_s': 13.4}
  2026-05-28-15-28-52|75     121.7 s | vggt_omega_512: ok | vggt_1b: ok
  2025-06-04-14-05-43|45     120.2 s | vggt_omega_512: ok | vggt_1b: ok
  2026-05-28-15-28-52|83     122.4 s | vggt_omega_512: ok | vggt_1b: ok
  2026-05-28-15-28-52|81     103.7 s | vggt_omega_512: ok | vggt_1b: ok
  2026-05-28-15-28-52|96     131.4 s | vggt_omega_512: ok | vggt_1b: ok
  2026-05-28-15-28-52|69     111.2 s | vggt_omega_512: ok | vggt_1b: ok
  2026-06-03-09-57-03|108    102.8 s | vggt_omega_512: ok | vggt_1b: ok
  2026-06-03-09-57-03|106    101.0 s | vggt_omega_512: ok | vggt_1b: ok
  2026-05-28-15-28-52|85      66.3 s | vggt_omega_512: ok | vggt_1b: ok
  2026-05-28-15-28-52|78      59.7 s | vggt_omega_512: ok | vggt_1b: ok
  2026-06-03-10-44-05|103     59.1 s | vggt_omega_512: ok | vggt_1b: ok
  2026-05-28-15-28-52|77      40.2 s | vggt_omega_512: ok | vggt_1b: ok


  fast unpack: {'archives': 1, 'xz_decompressed_on_all_cores': 0, 'download_s': 6.5, 'verify_and_decompress_s': 8.5, 'extract_s': 10.1}
  2026-06-03-10-44-05|97      59.9 s | vggt_omega_512: ok | vggt_1b: ok
  2026-06-02-17-05-20|105     55.3 s | vggt_omega_512: ok | vggt_1b: ok
  2026-06-02-17-05-20|104     56.1 s | vggt_omega_512: ok | vggt_1b: ok
  2026-06-03-10-44-05|114     62.9 s | vggt_omega_512: ok | vggt_1b: ok
  2026-06-02-17-05-20|112     55.7 s | vggt_omega_512: ok | vggt_1b: ok
  2026-06-03-10-44-05|108     59.4 s | vggt_omega_512: ok | vggt_1b: ok
  2026-06-02-17-05-20|107     57.6 s | vggt_omega_512: ok | vggt_1b: ok
  2026-06-03-10-44-05|99      59.2 s | vggt_omega_512: ok | vggt_1b: ok
  2026-06-02-17-05-20|106     55.7 s | vggt_omega_512: ok | vggt_1b: ok
  2026-06-03-10-44-05|101     57.2 s | vggt_omega_512: ok | vggt_1b: ok
  2026-06-02-17-05-20|115     63.5 s | vggt_omega_512: ok | vggt_1b: ok
  2026-06-03-10-44-05|100     53.6 s | vggt_omega_512: ok | vggt_1b: ok


  fast unpack: {'archives': 1, 'xz_decompressed_on_all_cores': 0, 'download_s': 3.1, 'verify_and_decompress_s': 5.3, 'extract_s': 6.1}
  2025-08-04-11-19-58|95     129.3 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|215    119.9 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|35     128.4 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|41     121.1 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|214    124.1 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|67     127.4 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|42     104.8 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|40     134.6 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|80     132.3 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|43      89.4 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|206    133.4 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-10-31-18|34     128.4 s | vggt_omega_512: ok | vggt_1b: ok
 

  fast unpack: {'archives': 1, 'xz_decompressed_on_all_cores': 0, 'download_s': 3.1, 'verify_and_decompress_s': 5.2, 'extract_s': 8.2}
  2025-08-04-11-19-58|85     105.5 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|39     113.8 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|34     107.8 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|84     100.0 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|38     101.5 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|52     105.2 s | vggt_omega_512: ok | vggt_1b: ok
  2026-06-18-10-56-11|24      84.8 s | vggt_omega_512: ok | vggt_1b: ok
  2026-06-18-10-37-06|52     112.1 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|93     102.4 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|77     114.0 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|90     101.6 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-10-31-18|43      99.9 s | vggt_omega_512: ok | vggt_1b: ok
 

  fast unpack: {'archives': 1, 'xz_decompressed_on_all_cores': 0, 'download_s': 2.3, 'verify_and_decompress_s': 3.9, 'extract_s': 4.5}
  2026-06-03-11-25-54|3       62.4 s | vggt_omega_512: ok | vggt_1b: ok
  2026-06-02-17-05-20|111     60.1 s | vggt_omega_512: ok | vggt_1b: ok
  2026-06-03-10-44-05|98      63.6 s | vggt_omega_512: ok | vggt_1b: ok
  2026-06-18-10-56-11|33      57.9 s | vggt_omega_512: ok | vggt_1b: ok
  2026-06-02-17-05-20|113     61.0 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|208    131.2 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|207    116.4 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|210    112.0 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|213    128.0 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|212    109.5 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-10-31-18|37     111.6 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|100    126.4 s | vggt_omega_512: ok | vggt_1b: ok
 

  fast unpack: {'archives': 1, 'xz_decompressed_on_all_cores': 0, 'download_s': 2.0, 'verify_and_decompress_s': 1.4, 'extract_s': 1.8}
  2025-06-12-14-23-14|400     54.5 s | vggt_omega_512: ok | vggt_1b: ok
  2025-06-13-09-11-24|112     49.1 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|199     62.1 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|180     63.1 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-12-58-34|1       63.3 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|133     54.3 s | vggt_omega_512: ok | vggt_1b: ok
  2025-08-04-11-19-58|209     89.8 s | vggt_omega_512: ok | vggt_1b: ok
  7 scenes with objects, 98 s


In [5]:
# --- 5. The answer ---
files = sorted(out_dir.glob(f"*_{tag}.csv"))
if not files:
    raise RuntimeError("cell 4 saved nothing: no scene had saved predictions and ground truth. Read its log above.")
rows = pd.concat([pd.read_csv(f, keep_default_na=False, na_values=[""]) for f in files], ignore_index=True)
rows["category"] = rows["category"].fillna("").astype(str)
print(len(files), "blocks |", rows["scene_id"].nunique(), "scenes | object categories by LiDAR pixels:")
census = rows[(rows["zone"] == "all") & (rows["depth_band"] == "all") & (rows["category"] != "all") & (rows["radius"] == RADII[0])
              & (rows["model"] == MODELS[0])].groupby(["category", "motion"])["n_pixels"].sum().unstack(fill_value=0)
print(census.sort_values(census.columns[0], ascending=False).head(12).to_string())


def show(title, table, digits=4):
    print()
    print("=====", title, "=====")
    print(table.round(digits).to_string(index=False) if len(table) else "(not enough scenes)")


report_dir = session.persist_root / "metrics" / RUN_TAG / "report_edges"
report_dir.mkdir(parents=True, exist_ok=True)
for model in MODELS:
    for radius in RADII:
        part = rows[(rows["model"] == model) & (rows["radius"] == radius)]
        print()
        print("#" * 30, model, "| boundary rim", radius, "px")
        show("AbsRel by motion and zone, all objects", edges.edge_summary(part))
        comparison = edges.edge_comparisons(part)
        show("paired over scenes", comparison)
        comparison.to_csv(report_dir / f"{model}_r{radius}_comparisons.csv", index=False)
        edges.edge_summary(part).to_csv(report_dir / f"{model}_r{radius}_summary.csv", index=False)
        if radius == RADII[0]:
            for band in ("1-10 m", "10-20 m", "20-40 m"):                     # same distance, so that depth is not the hidden cause
                show(f"paired over scenes, objects at {band} only", edges.edge_comparisons(part, depth_band=band))
            for category in census.sum(axis=1).sort_values(ascending=False).head(2).index:
                show(f"paired over scenes, category '{category}' only", edges.edge_comparisons(part, category=category))

6 blocks | 107 scenes | object categories by LiDAR pixels:
motion           moving   parked
category                        
car             3607924  9299765
bus              736691   338272
person           470942   254216
truck            236287   376662
dynamic          127800   357720
caravan          105743   556709
bicycle          105620   487247
trailer           72108    76542
bicyclist         50079    25236
portable-rider    10895     5257
portable           7143    45789
motorcycle         5763    31545

############################## vggt_omega_512 | boundary rim 6 px

===== AbsRel by motion and zone, all objects =====
    motion     zone  n_scenes    pixels  abs_rel  ci_low  ci_high  median_over_scenes
background      all       107 174388863   0.0769  0.0722   0.0820              0.0724
    moving      all       101   5538260   0.2029  0.1783   0.2346              0.1655
    moving boundary       101   2830255   0.2725  0.2410   0.3087              0.2246
    moving inter